In [1]:

import os, sys, time
print("=== SHELFSENSE LITE 11% — RETRAIN SAFE BOOT ===", flush=True)
print("[BOOT] Python:", sys.version, flush=True)
print("[BOOT] cwd:", os.getcwd(), flush=True)
print("[BOOT] heartbeat 1", flush=True); time.sleep(1)
print("[BOOT] heartbeat 2", flush=True)
open("/kaggle/working/SHELFSENSE_RETRAIN_STARTED.txt","w").write("started\n")
print("=== HEARTBEAT OK ===", flush=True)


=== SHELFSENSE LITE 11% — RETRAIN SAFE BOOT ===
[BOOT] Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
[BOOT] cwd: /kaggle/working
[BOOT] heartbeat 1
[BOOT] heartbeat 2
=== HEARTBEAT OK ===



# 11% — ShelfSense Lite
## SAFE RETRAIN NOTEBOOK — checkpoint first, evaluation later

This notebook intentionally does **one job only**:

> **retrain YOLO11s and persist the best checkpoint before the Kaggle runtime limit.**

The previous all-in-one notebook spent almost four hours training and then continued into validation, official test, counting threshold search, plotting, and packaging. That design was too risky for a Save & Run All job.

### New workflow

```text
Notebook A — THIS FILE
dataset audit
→ clean YOLO conversion
→ YOLO11s training
→ immediately copy best.pt
→ immediately create checkpoint ZIP
→ STOP

Notebook B — Final Evaluation
load best.pt
→ max_det=800 validation
→ official test
→ counting MAE/RMSE
→ plots
→ final package
```

### Why 36 epochs instead of 40?

The previous 40-epoch run consumed almost the whole Kaggle job window.  
This retrain uses **36 epochs** to leave safety margin for checkpoint persistence.

The goal here is not to fake the previous result. This is a new retrain and must be evaluated again afterward.


In [2]:

import importlib.util, subprocess, sys

need = []
if importlib.util.find_spec("ultralytics") is None:
    need.append("ultralytics>=8.4.0")
if importlib.util.find_spec("yaml") is None:
    need.append("pyyaml")

if need:
    print("[SETUP] Installing:", need, flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])

print("[SETUP] dependencies OK", flush=True)


[SETUP] Installing: ['ultralytics>=8.4.0']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 842.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 5.5 MB/s eta 0:00:00
[SETUP] dependencies OK


In [3]:

from pathlib import Path
import json, random, shutil, os, yaml, math
import numpy as np
import pandas as pd
import torch
from PIL import Image
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_ROOT = Path("/kaggle/input/datasets/wafabila/sku110k/SKU110K_fixed")
ANN_DIR = DATA_ROOT / "annotations"
IMG_DIR = DATA_ROOT / "images"

WORK = Path("/kaggle/working/shelfsense_retrain")
DS = WORK / "dataset_yolo"
REPORT = WORK / "reports"
RUNS = WORK / "runs"
SAFE = Path("/kaggle/working/shelfsense_checkpoint")

for p in [WORK, DS, REPORT, RUNS, SAFE]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("[CONFIG] DATA_ROOT:", DATA_ROOT)
print("[CONFIG] DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("[CONFIG] GPU:", torch.cuda.get_device_name(0))

assert DATA_ROOT.exists(), DATA_ROOT
assert ANN_DIR.exists(), ANN_DIR
assert IMG_DIR.exists(), IMG_DIR


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
[CONFIG] DATA_ROOT: /kaggle/input/datasets/wafabila/sku110k/SKU110K_fixed
[CONFIG] DEVICE: 0
[CONFIG] GPU: Tesla T4


## 1. Audit official SKU-110K structure

In [4]:

img_files = [
    p for p in IMG_DIR.iterdir()
    if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"}
]

COLS = ["image","x1","y1","x2","y2","source_class","image_width","image_height"]

csvs = {
    "train": ANN_DIR / "annotations_train.csv",
    "val": ANN_DIR / "annotations_val.csv",
    "test": ANN_DIR / "annotations_test.csv",
}

frames = {}
for split, p in csvs.items():
    df = pd.read_csv(p, names=COLS, header=None)
    frames[split] = df
    print(f"[{split}] rows={len(df):,} images={df['image'].nunique():,}")

print("[DATA] mounted images:", len(img_files))

expected = {"train":8219, "val":588, "test":2936}
for split, n in expected.items():
    actual = int(frames[split]["image"].nunique())
    assert actual == n, f"{split}: expected {n}, got {actual}"

assert len(img_files) == 11743, f"Expected 11743 images, got {len(img_files)}"
print("[AUDIT] official split counts OK")


[train] rows=1,208,482 images=8,219
[val] rows=90,968 images=588
[test] rows=431,546 images=2,936
[DATA] mounted images: 11743
[AUDIT] official split counts OK


In [5]:

available = {p.name: p for p in img_files}

audit = {}
for split, df in frames.items():
    numeric = df[["x1","y1","x2","y2","image_width","image_height"]].apply(
        pd.to_numeric, errors="coerce"
    )
    bad_nan = numeric.isna().any(axis=1)
    bad_geom = (numeric["x2"] <= numeric["x1"]) | (numeric["y2"] <= numeric["y1"])
    bad_bounds = (
        (numeric["x1"] < 0) |
        (numeric["y1"] < 0) |
        (numeric["x2"] > numeric["image_width"]) |
        (numeric["y2"] > numeric["image_height"])
    )
    missing = ~df["image"].astype(str).isin(available.keys())
    bad = bad_nan | bad_geom | bad_bounds | missing

    audit[split] = {
        "images": int(df["image"].nunique()),
        "rows": int(len(df)),
        "invalid_rows": int(bad.sum()),
    }

print(json.dumps(audit, indent=2))
(REPORT / "source_audit.json").write_text(json.dumps(audit, indent=2))


{
  "train": {
    "images": 8219,
    "rows": 1208482,
    "invalid_rows": 14918
  },
  "val": {
    "images": 588,
    "rows": 90968,
    "invalid_rows": 1031
  },
  "test": {
    "images": 2936,
    "rows": 431546,
    "invalid_rows": 5211
  }
}


248

## 2. Convert to single-class YOLO detection labels

In [6]:

if DS.exists():
    shutil.rmtree(DS)

for split in ["train","val","test"]:
    (DS/"images"/split).mkdir(parents=True, exist_ok=True)
    (DS/"labels"/split).mkdir(parents=True, exist_ok=True)

conversion = {}

for split, df0 in frames.items():
    df = df0.copy()

    for c in ["x1","y1","x2","y2","image_width","image_height"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    valid = (
        df[["x1","y1","x2","y2","image_width","image_height"]].notna().all(axis=1) &
        (df["x2"] > df["x1"]) &
        (df["y2"] > df["y1"]) &
        (df["x1"] >= 0) &
        (df["y1"] >= 0) &
        (df["x2"] <= df["image_width"]) &
        (df["y2"] <= df["image_height"]) &
        df["image"].astype(str).isin(available.keys())
    )

    clean = df.loc[valid].copy()
    n_img = 0
    n_box = 0

    for image_name, g in clean.groupby("image", sort=False):
        src = available[str(image_name)]
        dst = DS/"images"/split/src.name

        if not dst.exists():
            os.symlink(src, dst)

        label_path = (DS/"labels"/split/src.stem).with_suffix(".txt")
        lines = []

        for _, r in g.iterrows():
            w = float(r.image_width)
            h = float(r.image_height)

            xc = ((float(r.x1)+float(r.x2))/2) / w
            yc = ((float(r.y1)+float(r.y2))/2) / h
            bw = (float(r.x2)-float(r.x1)) / w
            bh = (float(r.y2)-float(r.y1)) / h

            xc = np.clip(xc,0,1)
            yc = np.clip(yc,0,1)
            bw = np.clip(bw,1e-8,1)
            bh = np.clip(bh,1e-8,1)

            lines.append(f"0 {xc:.7f} {yc:.7f} {bw:.7f} {bh:.7f}")

        label_path.write_text("\n".join(lines) + "\n")
        n_img += 1
        n_box += len(lines)

    conversion[split] = {
        "images": n_img,
        "boxes": n_box,
        "dropped_rows": int((~valid).sum()),
    }

print(json.dumps(conversion, indent=2))
(REPORT / "conversion_summary.json").write_text(json.dumps(conversion, indent=2))


{
  "train": {
    "images": 8219,
    "boxes": 1193564,
    "dropped_rows": 14918
  },
  "val": {
    "images": 588,
    "boxes": 89937,
    "dropped_rows": 1031
  },
  "test": {
    "images": 2936,
    "boxes": 426335,
    "dropped_rows": 5211
  }
}


251

In [7]:

data_yaml = WORK / "sku110k.yaml"
data_yaml.write_text(yaml.safe_dump({
    "path": str(DS),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 1,
    "names": ["object"],
}, sort_keys=False))

print(data_yaml.read_text())


path: /kaggle/working/shelfsense_retrain/dataset_yolo
train: images/train
val: images/val
test: images/test
nc: 1
names:
- object



## 3. Post-conversion integrity check

In [8]:

def check_split(split):
    labels = sorted((DS/"labels"/split).glob("*.txt"))
    images = sorted((DS/"images"/split).iterdir())
    bad = 0
    boxes = 0

    for lp in labels:
        for line in lp.read_text().splitlines():
            if not line.strip():
                continue
            parts = line.split()
            if len(parts) != 5:
                bad += 1
                continue
            try:
                cls, x, y, w, h = map(float, parts)
                ok = (
                    cls == 0 and
                    0 <= x <= 1 and
                    0 <= y <= 1 and
                    0 < w <= 1 and
                    0 < h <= 1
                )
                if not ok:
                    bad += 1
                boxes += 1
            except:
                bad += 1

    return {
        "images": len(images),
        "labels": len(labels),
        "boxes": boxes,
        "bad_label_lines": bad,
    }

converted_audit = {s: check_split(s) for s in ["train","val","test"]}
print(json.dumps(converted_audit, indent=2))

assert sum(v["bad_label_lines"] for v in converted_audit.values()) == 0
print("[AUDIT] conversion OK")


{
  "train": {
    "images": 8219,
    "labels": 8219,
    "boxes": 1193564,
    "bad_label_lines": 0
  },
  "val": {
    "images": 588,
    "labels": 588,
    "boxes": 89937,
    "bad_label_lines": 0
  },
  "test": {
    "images": 2936,
    "labels": 2936,
    "boxes": 426335,
    "bad_label_lines": 0
  }
}
[AUDIT] conversion OK



## 4. Safety checkpoint callback

After every fit epoch, the notebook attempts to copy the current:

```text
best.pt
last.pt
```

into:

```text
/kaggle/working/shelfsense_checkpoint/
```

This is deliberately outside the training run directory so the checkpoint is easy to find in Kaggle Output.


In [9]:

def persist_checkpoint(trainer):
    try:
        safe_dir = Path("/kaggle/working/shelfsense_checkpoint")
        safe_dir.mkdir(parents=True, exist_ok=True)

        best = Path(str(trainer.best))
        last = Path(str(trainer.last))

        if best.exists():
            shutil.copy2(best, safe_dir / "shelfsense_best.pt")

        if last.exists():
            shutil.copy2(last, safe_dir / "shelfsense_last.pt")

        status = {
            "epoch_completed": int(getattr(trainer, "epoch", -1)) + 1,
            "best_exists": best.exists(),
            "last_exists": last.exists(),
            "best_source": str(best),
            "last_source": str(last),
        }

        (safe_dir / "checkpoint_status.json").write_text(
            json.dumps(status, indent=2)
        )

        print(
            f"[SAFE-CHECKPOINT] epoch={status['epoch_completed']} "
            f"best={status['best_exists']} last={status['last_exists']}",
            flush=True
        )

    except Exception as e:
        print("[SAFE-CHECKPOINT][WARN]", repr(e), flush=True)



## 5. Retrain YOLO11s

Configuration stays close to the previous run:

- YOLO11s
- 960 px
- batch 4
- AdamW
- cosine LR
- AMP
- conservative dense-shelf augmentation

Changes for safety:

- **36 epochs** instead of 40;
- `patience=8`;
- no heavy final evaluation in this notebook;
- checkpoint copied outside the training directory after every epoch.


In [10]:

MODEL_NAME = "yolo11s.pt"

model = YOLO(MODEL_NAME)
model.add_callback("on_fit_epoch_end", persist_checkpoint)

train_result = model.train(
    data=str(data_yaml),

    epochs=36,
    imgsz=960,
    batch=4,
    device=DEVICE,
    workers=2,

    patience=8,
    pretrained=True,

    optimizer="AdamW",
    lr0=5e-4,
    lrf=0.05,
    weight_decay=5e-4,
    cos_lr=True,
    warmup_epochs=3.0,

    amp=True,
    cache=False,

    seed=SEED,
    deterministic=True,

    plots=False,

    hsv_h=0.008,
    hsv_s=0.30,
    hsv_v=0.20,
    degrees=2.0,
    translate=0.04,
    scale=0.15,
    shear=0.0,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,

    mosaic=0.35,
    mixup=0.0,
    close_mosaic=8,

    max_det=800,

    project=str(RUNS),
    name="yolo11s_960_retrain",
    exist_ok=True,
    verbose=True,
)

TRAIN_DIR = Path(train_result.save_dir)
BEST_PT = TRAIN_DIR / "weights" / "best.pt"
LAST_PT = TRAIN_DIR / "weights" / "last.pt"

print("[TRAIN] save_dir:", TRAIN_DIR)
print("[TRAIN] best:", BEST_PT, BEST_PT.exists())
print("[TRAIN] last:", LAST_PT, LAST_PT.exists())

assert BEST_PT.exists(), f"best.pt missing: {BEST_PT}"


Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=8, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/shelfsense_retrain/sku110k.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=36, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.008, hsv_s=0.3, hsv_v=0.2, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.05, mask_ratio=4, max_det=800, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=0.35, multi_scale=0.0, name=yolo11s_960_retrain, nbs=64, nms=None, ops

Corrupt JPEG data: premature end of data segment


       1/36      3.28G      1.604      1.092      1.217        667        960: 15% ━╸────────── 316/2054 5.5it/s 58.5s<5:14

Corrupt JPEG data: 102 extraneous bytes before marker 0xd9


       1/36      3.54G      1.599      1.074      1.212        849        960: 17% ━━────────── 350/2054 6.3it/s 1:04<4:30

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 79% ━━━━━━━━━╸── 59/74 4.6it/s 20.5s<3.3s

Corrupt JPEG data: premature end of data segment


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 98% ━━━━━━━━━━━╸ 73/74 3.7it/s 27.7s<0.3s

Corrupt JPEG data: 786 extraneous bytes before marker 0xd9


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 74/74 2.4it/s 30.9s
                   all        588      89937      0.912      0.873      0.936      0.585
Speed: 0.3ms preprocess, 32.6ms inference, 0.0ms loss, 1.9ms postprocess per image
[SAFE-CHECKPOINT] epoch=37 best=True last=True
[TRAIN] save_dir: /kaggle/working/shelfsense_retrain/runs/yolo11s_960_retrain
[TRAIN] best: /kaggle/working/shelfsense_retrain/runs/yolo11s_960_retrain/weights/best.pt True
[TRAIN] last: /kaggle/working/shelfsense_retrain/runs/yolo11s_960_retrain/weights/last.pt True


## 6. Immediately persist final checkpoint

In [11]:

SAFE.mkdir(parents=True, exist_ok=True)

FINAL_BEST = SAFE / "shelfsense_yolo11s_best.pt"
FINAL_LAST = SAFE / "shelfsense_yolo11s_last.pt"

shutil.copy2(BEST_PT, FINAL_BEST)

if LAST_PT.exists():
    shutil.copy2(LAST_PT, FINAL_LAST)

# Preserve lightweight training metadata if available
for name in ["results.csv", "args.yaml"]:
    src = TRAIN_DIR / name
    if src.exists():
        shutil.copy2(src, SAFE / name)

print("[SAFE] best:", FINAL_BEST, FINAL_BEST.stat().st_size/1024/1024, "MB")
if FINAL_LAST.exists():
    print("[SAFE] last:", FINAL_LAST, FINAL_LAST.stat().st_size/1024/1024, "MB")


[SAFE] best: /kaggle/working/shelfsense_checkpoint/shelfsense_yolo11s_best.pt 18.302026748657227 MB
[SAFE] last: /kaggle/working/shelfsense_checkpoint/shelfsense_yolo11s_last.pt 18.302026748657227 MB



## 7. Tiny smoke test only

This is **not** the final benchmark.

The goal is just to verify the persisted custom checkpoint can be loaded and can run inference.

Final validation/test metrics belong in the separate evaluation notebook.


In [12]:

check_model = YOLO(str(FINAL_BEST))

sample_images = sorted((DS/"images"/"val").iterdir())[:2]

smoke = check_model.predict(
    source=[str(p) for p in sample_images],
    imgsz=960,
    conf=0.25,
    iou=0.6,
    max_det=800,
    device=DEVICE,
    verbose=False,
)

print("[SMOKE] predictions:", len(smoke))
for p, r in zip(sample_images, smoke):
    n = 0 if r.boxes is None else len(r.boxes)
    print(p.name, "detections=", n)


[SMOKE] predictions: 2
val_0.jpg detections= 154
val_1.jpg detections= 169


## 8. Create checkpoint ZIP immediately

In [13]:

manifest = {
    "project": "ShelfSense Lite",
    "portfolio_progress": "11%",
    "stage": "retrain checkpoint only",
    "model": "YOLO11s",
    "imgsz": 960,
    "epochs_requested": 36,
    "checkpoint": str(FINAL_BEST),
    "dataset": str(DATA_ROOT),
    "next_step": "Run separate final evaluation notebook with max_det=800.",
}

(SAFE / "retrain_manifest.json").write_text(
    json.dumps(manifest, indent=2)
)

ZIP_PATH = shutil.make_archive(
    "/kaggle/working/shelfsense_RETRAIN_CHECKPOINT",
    "zip",
    root_dir=str(SAFE),
)

print("\n=== SHELFSENSE RETRAIN CHECKPOINT SAVED ===", flush=True)
print("[DONE] BEST:", FINAL_BEST, flush=True)
print("[DONE] ZIP:", ZIP_PATH, flush=True)
print("[DONE] NEXT: attach this output to the final-evaluation notebook.", flush=True)



=== SHELFSENSE RETRAIN CHECKPOINT SAVED ===
[DONE] BEST: /kaggle/working/shelfsense_checkpoint/shelfsense_yolo11s_best.pt
[DONE] ZIP: /kaggle/working/shelfsense_RETRAIN_CHECKPOINT.zip
[DONE] NEXT: attach this output to the final-evaluation notebook.



# Stop Here

If this cell prints:

```text
=== SHELFSENSE RETRAIN CHECKPOINT SAVED ===
[DONE] BEST: /kaggle/working/shelfsense_checkpoint/shelfsense_yolo11s_best.pt
[DONE] ZIP: /kaggle/working/shelfsense_RETRAIN_CHECKPOINT.zip
```

then **do not add official test/counting evaluation to this notebook**.

Download or attach the output to the separate final-evaluation notebook.

Expected persistent outputs:

```text
/kaggle/working/shelfsense_checkpoint/shelfsense_yolo11s_best.pt
/kaggle/working/shelfsense_RETRAIN_CHECKPOINT.zip
```

This notebook intentionally stops after checkpoint persistence so the 4-hour Save Version failure pattern is not repeated.
